# ChartQA v1.0 baselines (Week 1 gate)

Runs a base instruction model on the ChartAudit v1.0 **test** split (180 items, never trained on), zero-shot and few-shot, and scores it.

**Runtime:** `Runtime → Change runtime type → T4 GPU` (free tier is enough; models load in 4-bit). One model on the test split takes roughly 10–20 minutes on a T4.

**Secrets** (key icon in the left sidebar, turn on "Notebook access"):
- `GITHUB_TOKEN`: a GitHub token with read access to `parulrewatkar0692/chartqa-capstone` (the repo is private). Without it, upload `test.jsonl` and `train.jsonl` by hand (cell 2 explains).
- `HF_TOKEN`: a Hugging Face token. Llama 3.1 and Mistral are gated: accept their licence on the model page first.

**Metrics**
- **Relaxed accuracy**: numbers within 5% of gold (the ChartQA convention), text and yes/no after normalising case and punctuation, unanswerable items correct when the model returns `"answer": null`.
- **JSON valid**: the output parses as JSON. **Schema valid**: it has all five fields with allowed `answer_type`.
- **Evidence grounded**: share of the model's evidence strings that point at real cells/colours/title in the input (same checker as the teacher labels).

In [ ]:
# 1. Install
!pip -q install -U "transformers>=4.45" accelerate bitsandbytes

In [ ]:
# 2. Settings
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"   # or "meta-llama/Llama-3.1-8B-Instruct", "mistralai/Mistral-7B-Instruct-v0.3"
SPLIT = "test"          # "test" for the reported baseline, "val" while you experiment
N_SHOTS = [0, 3]        # zero-shot and 3-shot; few-shot examples come from train only
LIMIT = None            # e.g. 20 for a quick smoke test
BATCH_SIZE = 8
MAX_NEW_TOKENS = 256
LOAD_IN_4BIT = True
SAVE_TO_DRIVE = True    # results go to MyDrive/chartqa_baselines/
SEED = 42

In [ ]:
# 3. Get the dataset (clones the private repo with GITHUB_TOKEN; falls back to manual upload)
import os, subprocess
REPO = "parulrewatkar0692/chartqa-capstone"
DATA_DIR = "/content/chartqa-capstone/dataset"
if not os.path.isdir(DATA_DIR):
    token = None
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    except Exception as e:
        print("No GITHUB_TOKEN secret:", e)
    if token:
        subprocess.run(["git", "clone", "--depth", "1", f"https://{token}@github.com/{REPO}.git",
                        "/content/chartqa-capstone"], check=True)
    else:
        # Manual route: upload dataset/v1.0/test.jsonl, train.jsonl (and val.jsonl) plus
        # dataset/teacher/check_teacher.py from the repo.
        from google.colab import files
        os.makedirs(f"{DATA_DIR}/v1.0", exist_ok=True); os.makedirs(f"{DATA_DIR}/teacher", exist_ok=True)
        for name, data in files.upload().items():
            dest = f"{DATA_DIR}/teacher/{name}" if name.endswith(".py") else f"{DATA_DIR}/v1.0/{name}"
            open(dest, "wb").write(data)
print(sorted(os.listdir(f"{DATA_DIR}/v1.0")))

In [ ]:
# 4. Load the splits
import json, random, sys, collections
def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

eval_set = read_jsonl(f"{DATA_DIR}/v1.0/{SPLIT}.jsonl")
train = read_jsonl(f"{DATA_DIR}/v1.0/train.jsonl")
if LIMIT:
    eval_set = eval_set[:LIMIT]
sys.path.insert(0, f"{DATA_DIR}/teacher")
import check_teacher as CT   # evidence checker used for the teacher labels
print(len(eval_set), "eval items |", collections.Counter(r["answer_type"] for r in eval_set))

In [ ]:
# 5. Prompts. The eval item's system + user turns are used as-is; few-shot examples are
#    full train conversations (question -> grounded JSON answer) placed before it.
def pick_shots(train, k, seed=SEED):
    # One example per answer type first (number, text, yes_no, unanswerable), short inputs only.
    rng = random.Random(seed)
    pool = [r for r in train if len(r["input_text"]) < 900]
    rng.shuffle(pool)
    order = ["number", "text", "yes_no", "unanswerable", "number", "text"]
    shots, used = [], set()
    for t in order:
        for r in pool:
            if r["answer_type"] == t and r["id"] not in used:
                shots.append(r); used.add(r["id"]); break
        if len(shots) == k:
            break
    return shots

def build_messages(item, shots):
    msgs = [item["messages"][0]]
    for s in shots:
        msgs += [s["messages"][1], s["messages"][2]]
    msgs.append(item["messages"][1])
    return msgs

def to_prompt(tokenizer, msgs):
    try:
        return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    except Exception:
        # Templates without a system role (e.g. Mistral): fold the system prompt into the first user turn.
        first = dict(msgs[1]); first["content"] = msgs[0]["content"] + "\n\n" + first["content"]
        return tokenizer.apply_chat_template([first] + msgs[2:], tokenize=False, add_generation_prompt=True)

In [ ]:
# 6. Parse and score
import re
def extract_json(text):
    text = text.strip()
    text = re.sub(r"^```(?:json)?\s*|\s*```$", "", text)
    start = text.find("{")
    if start < 0:
        return None
    depth, in_str, esc = 0, False, False
    for i, ch in enumerate(text[start:], start):
        if in_str:
            esc = (ch == "\\") and not esc
            if ch == '"' and not esc:
                in_str = False
            continue
        if ch == '"':
            in_str = True
        elif ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                try:
                    return json.loads(text[start:i + 1])
                except json.JSONDecodeError:
                    return None
    return None

FIELDS = {"answer", "answer_type", "unit", "evidence", "rationale"}
TYPES = {"number", "text", "yes_no", "unanswerable"}

def relaxed_match(pred, gold, answer_type):
    if answer_type == "unanswerable":
        return pred is None or (isinstance(pred, str) and pred.strip().lower() in ("", "null", "none"))
    if pred is None:
        return False
    if answer_type == "number":
        p, g = CT.num(pred), CT.num(gold)
        if p is None or g is None:
            return False
        return abs(p - g) <= 0.05 * abs(g) if g != 0 else abs(p) < 1e-9
    return CT.norm(pred) == CT.norm(gold)

def score(item, raw):
    out = extract_json(raw)
    gold = json.loads(item["messages"][2]["content"])
    res = {"id": item["id"], "raw": raw, "json_valid": out is not None}
    res["schema_valid"] = bool(out) and set(out) >= FIELDS and out.get("answer_type") in TYPES \
        and isinstance(out.get("evidence"), list)
    res["pred"] = out.get("answer") if out else None
    res["correct"] = bool(out) and relaxed_match(res["pred"], gold["answer"], item["answer_type"])
    ev = [e for e in (out or {}).get("evidence") or [] if isinstance(e, str)]
    res["evidence_n"] = len(ev)
    res["evidence_ok"] = sum(CT.check_evidence(e, item["input_text"]) is None for e in ev)
    return res

In [ ]:
# 7. Load the model
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
try:
    from google.colab import userdata
    from huggingface_hub import login
    login(userdata.get("HF_TOKEN"))
except Exception as e:
    print("HF login skipped:", e)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                           bnb_4bit_compute_dtype=torch.float16) if LOAD_IN_4BIT else None
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, device_map="auto", quantization_config=quant,
                                             torch_dtype=torch.float16)
model.eval()
print(MODEL_ID, "loaded on", model.device)

In [ ]:
# 8. Generate (greedy) and score
from tqdm.auto import tqdm

@torch.no_grad()
def generate(prompts):
    outs = []
    for i in tqdm(range(0, len(prompts), BATCH_SIZE)):
        batch = tokenizer(prompts[i:i + BATCH_SIZE], return_tensors="pt", padding=True,
                          add_special_tokens=False).to(model.device)
        gen = model.generate(**batch, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                             pad_token_id=tokenizer.pad_token_id)
        outs += tokenizer.batch_decode(gen[:, batch["input_ids"].shape[1]:], skip_special_tokens=True)
    return outs

runs = {}
for k in N_SHOTS:
    shots = pick_shots(train, k)
    prompts = [to_prompt(tokenizer, build_messages(it, shots)) for it in eval_set]
    raws = generate(prompts)
    runs[k] = [score(it, raw) for it, raw in zip(eval_set, raws)]
    acc = sum(r["correct"] for r in runs[k]) / len(runs[k])
    print(f"{k}-shot relaxed accuracy: {acc:.3f}")

In [ ]:
# 9. Report
import pandas as pd

def summarise(results, items):
    df = pd.DataFrame(results).merge(
        pd.DataFrame([{k: it[k] for k in ("id", "source", "chart_type", "answer_type", "question_type")}
                      for it in items]), on="id")
    overall = {
        "n": len(df),
        "relaxed_accuracy": df.correct.mean(),
        "json_valid": df.json_valid.mean(),
        "schema_valid": df.schema_valid.mean(),
        "evidence_grounded": df.evidence_ok.sum() / max(df.evidence_n.sum(), 1),
        "refusal_accuracy": df[df.answer_type == "unanswerable"].correct.mean(),
        "answerable_accuracy": df[df.answer_type != "unanswerable"].correct.mean(),
    }
    by = {col: df.groupby(col).correct.agg(["mean", "count"]).round(3) for col in
          ("question_type", "answer_type", "source", "chart_type")}
    return df, overall, by

summary_rows = []
for k, res in runs.items():
    df, overall, by = summarise(res, eval_set)
    summary_rows.append({"model": MODEL_ID, "split": SPLIT, "shots": k, **{a: round(b, 4) for a, b in overall.items()}})
    print(f"\n===== {MODEL_ID} | {SPLIT} | {k}-shot =====")
    print(pd.Series(overall).round(3).to_string())
    for col, t in by.items():
        print(f"\nby {col}:\n{t.to_string()}")
summary = pd.DataFrame(summary_rows)
summary

In [ ]:
# 10. Save predictions and the summary (Drive if available, else /content)
import datetime
out_dir = "/content/chartqa_baselines"
if SAVE_TO_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        out_dir = "/content/drive/MyDrive/chartqa_baselines"
    except Exception as e:
        print("Drive not mounted, saving locally:", e)
os.makedirs(out_dir, exist_ok=True)
tag = f"{MODEL_ID.split('/')[-1]}_{SPLIT}_{datetime.datetime.now():%Y%m%d-%H%M}"
for k, res in runs.items():
    with open(f"{out_dir}/{tag}_{k}shot_predictions.jsonl", "w") as f:
        for r in res:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
summary_path = f"{out_dir}/summary.csv"
summary.to_csv(summary_path, mode="a", header=not os.path.exists(summary_path), index=False)
print("saved to", out_dir)

In [ ]:
# 11. Free GPU memory before switching MODEL_ID in cell 2 and re-running cells 7-10
import gc
del model; gc.collect(); torch.cuda.empty_cache()